In [ ]:
import os,sys,json,subprocess,hashlib
from pathlib import Path
base=Path('/kaggle/working/urop_mm_part2_20261027');base.mkdir(parents=True,exist_ok=True)
(base/'evaluate_frozen_multimodal_v1.py').write_text('"""Evaluate one frozen, training-complete checkpoint; never optimize or select on test."""\nimport argparse,datetime,hashlib,json,os,sys,time\nfrom pathlib import Path\n\ndef now():return datetime.datetime.now(datetime.timezone.utc).isoformat()\ndef run(a):\n import numpy as np,pandas as pd,torch\n import train_usable_multimodal_v2 as t\n from usable_cohort_v2 import load_input\n from multimodal_recovery_v1 import sha,atomic_json,load_generation\n from verify_multimodal_final_v1 import verify,LABELS\n from sklearn.metrics import classification_report,confusion_matrix\n folder=Path(a.run);protocol=json.loads((folder/\'protocol.json\').read_text());cfg=protocol[\'config\']\n payload,proof=load_generation(folder/\'recovery\');state=payload[\'state\'];assert payload[\'contract\']==protocol\n assert state[\'epoch\']>4 or state[\'early_stopped\'];assert len(state[\'history\'])>=3\n selected=max(state[\'history\'],key=lambda h:(h[\'validation\'][\'macro_f1\'],-h[\'validation\'][\'loss\'],-h[\'epoch\']))\n assert selected[\'epoch\']==state[\'best\'][\'epoch\'];best=state[\'best\'];weights=folder/best[\'weights_file\'];assert sha(weights)==best[\'weights_sha256\']\n freeze=folder/\'final_selection_frozen.json\'\n value={\'utc\':now(),\'state\':\'frozen_before_test\',\'selection_rule\':cfg[\'selection\'],\'selected\':best,\'protocol_sha256\':sha(folder/\'protocol.json\'),\'source_hashes\':protocol[\'sources\'],\'inputs\':protocol[\'inputs\'],\'training_complete\':True,\'recovery\':proof}\n if freeze.exists():\n  existing=json.loads(freeze.read_text());assert existing[\'selected\']==best and existing[\'protocol_sha256\']==value[\'protocol_sha256\']\n else:atomic_json(freeze,value)\n expected,_=load_input(a.root);official=pd.DataFrame([r for r in expected.values() if r[\'split\']==\'test\']);assert len(official)==59270\n manifest=pd.read_csv(Path(a.root)/\'data/official_paired_v2/paired_manifest.csv\',dtype=str,keep_default_na=False);assert sha(Path(a.root)/\'data/official_paired_v2/paired_manifest.csv\')==protocol[\'inputs\'][\'manifest_sha256\']\n test=manifest[manifest.split.eq(\'test\')].set_index(\'id\').loc[official.id].reset_index()\n for key in [\'clean_title\',\'6_way_label\']:assert np.array_equal(test[key].astype(str).to_numpy(),official[key].astype(str).to_numpy())\n if (folder/\'test_metrics.json\').exists():\n  result=verify(folder,\'test\',official,\'test\');atomic_json(folder/\'test_independent_verification.json\',result);return result\n assert not (folder/\'test_start.json\').exists(),\'Interrupted test requires explicitly reviewed replay; never silently re-run\'\n atomic_json(folder/\'test_start.json\',{\'utc\':now(),\'weights_sha256\':best[\'weights_sha256\'],\'selection_receipt_sha256\':sha(freeze),\'expected_rows\':59270,\'policy\':\'one full frozen evaluation; no optimizer or tuning\'})\n tokenizer=t.AutoTokenizer.from_pretrained(folder/\'tokenizer\');bcfg=t.AutoConfig.from_pretrained(cfg[\'bert_checkpoint\'],revision=cfg[\'bert_revision\'])\n model=t.MaximumFusion(t.BertModel(bcfg,add_pooling_layer=False),t.resnet50(weights=None));model.image.fc=t.nn.Identity()\n loaded=torch.load(weights,map_location=\'cpu\',weights_only=True);model.load_state_dict(loaded,strict=True);model=model.cuda().eval()\n # Same selected tensor, validation-only probe: strict reload roundtrip and logits.\n val=manifest[manifest.split.eq(\'validation\')].iloc[:32].copy();ds=t.Pairs(val,a.root,tokenizer,cfg,False,a.index,protocol[\'inputs\'][\'packed_index_sha256\']);batch=next(iter(t.build_loader(ds,tokenizer,cfg,False)));batch.pop(\'ids\');batch.pop(\'labels\')\n with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.float16):before=model(**{k:v.cuda() for k,v in batch.items()})\n model.load_state_dict(loaded,strict=True)\n with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.float16):after=model(**{k:v.cuda() for k,v in batch.items()})\n assert torch.equal(before,after);atomic_json(folder/\'selected_weight_roundtrip.json\',{\'utc\':now(),\'sha256\':sha(weights),\'strict_load\':True,\'validation_probe_rows\':len(val),\'maximum_logit_delta\':float((before-after).abs().max())})\n del loaded,ds,batch,before,after\n started=time.monotonic();data=t.Pairs(test,a.root,tokenizer,cfg,False,a.index,protocol[\'inputs\'][\'packed_index_sha256\']);metrics,frame=t.evaluate(model,t.build_loader(data,tokenizer,cfg,False),torch.device(\'cuda\'));assert frame.id.tolist()==official.id.tolist();frame.insert(1,\'split\',\'test\')\n frame.to_csv(folder/\'test_predictions.csv\',index=False,float_format=\'%.17g\');atomic_json(folder/\'test_metrics.json\',metrics)\n y=frame.true_label.to_numpy();pred=frame.predicted_label.to_numpy();cm=confusion_matrix(y,pred,labels=range(6));atomic_json(folder/\'test_classification_report.json\',classification_report(y,pred,labels=range(6),target_names=LABELS,zero_division=0,output_dict=True));atomic_json(folder/\'test_confusions.json\',{\'labels\':LABELS,\'raw\':cm.tolist(),\'row_normalized\':(cm/cm.sum(1,keepdims=True)).tolist()})\n result=verify(folder,\'test\',official,\'test\');result.update(utc=now(),inference_seconds=time.monotonic()-started,selected_epoch=best[\'epoch\'],weights_sha256=best[\'weights_sha256\']);atomic_json(folder/\'test_independent_verification.json\',result)\n # Preserve original selected validation file; make a labelled final export copy.\n prefix=\'validation_epoch\'+str(best[\'epoch\']);vf=pd.read_csv(folder/(prefix+\'_predictions.csv\'),dtype={\'id\':str});vf.insert(1,\'split\',\'validation\');vf.to_csv(folder/\'validation_predictions.csv\',index=False,float_format=\'%.17g\')\n import shutil\n for suffix in [\'metrics.json\',\'classification_report.json\',\'confusions.json\']:shutil.copyfile(folder/(prefix+\'_\'+suffix),folder/(\'validation_\'+suffix))\n voff=pd.DataFrame([r for r in expected.values() if r[\'split\']==\'validation\']);atomic_json(folder/\'validation_independent_verification.json\',verify(folder,\'validation\',voff,\'validation\'))\n atomic_json(folder/\'final_evaluation_complete.json\',{\'utc\':now(),\'test\':result,\'final_selected_epoch\':best[\'epoch\'],\'state\':\'runtime_verified_requires_saved_private_and_local_readback\'})\n return result\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--run\',required=True,type=Path);p.add_argument(\'--root\',required=True,type=Path);p.add_argument(\'--index\',required=True,type=Path);a=p.parse_args();print(json.dumps(run(a)),flush=True)\n')
(base/'verify_multimodal_final_v1.py').write_text('"""Independent CSV-only verification for a frozen multimodal validation/test.\nNo model, inference, training or network access. Output is a returned dict.\n"""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,classification_report,confusion_matrix\nLABELS=[\'True\',\'Satire/Parody\',\'Misleading Content\',\'Imposter Content\',\'False Connection\',\'Manipulated Content\']\ndef sha(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for b in iter(lambda:f.read(4*1024**2),b\'\'):h.update(b)\n return h.hexdigest()\ndef verify(folder,prefix,official,split):\n folder=Path(folder);frame=pd.read_csv(folder/(prefix+\'_predictions.csv\'),dtype={\'id\':str})\n assert len(frame)==len(official) and frame.id.is_unique and frame.id.tolist()==official.id.tolist()\n assert frame[\'split\'].eq(split).all()\n y=frame.true_label.to_numpy();p=frame.predicted_label.to_numpy();ce=frame.cross_entropy.to_numpy();probs=frame[[\'p\'+str(i) for i in range(6)]].to_numpy()\n assert np.array_equal(y,official[\'6_way_label\'].astype(int).to_numpy())\n assert np.isfinite(probs).all() and np.isfinite(ce).all() and (ce>=0).all() and (probs>=0).all() and (probs<=1).all()\n assert np.allclose(probs.sum(1),1,atol=2e-6,rtol=0) and np.array_equal(probs.argmax(1),p)\n gold=probs[np.arange(len(y)),y];positive=gold>0\n assert np.allclose(-np.log(gold[positive]),ce[positive],rtol=2e-5,atol=2e-5)\n assert (ce[~positive]>=-np.log(np.finfo(np.float32).tiny)).all()\n metrics={\'accuracy\':float(accuracy_score(y,p)),\'micro_f1\':float(f1_score(y,p,average=\'micro\')),\'macro_f1\':float(f1_score(y,p,average=\'macro\',labels=range(6),zero_division=0)),\'weighted_f1\':float(f1_score(y,p,average=\'weighted\',labels=range(6),zero_division=0)),\'balanced_accuracy\':float(balanced_accuracy_score(y,p)),\'loss\':float(ce.mean()),\'samples\':len(frame)}\n def compare(a,b):\n  if isinstance(a,dict):\n   assert set(a)==set(b)\n   for k in a:compare(a[k],b[k])\n  else:assert abs(a-b)<1e-10\n compare(json.loads((folder/(prefix+\'_metrics.json\')).read_text()),metrics)\n compare(json.loads((folder/(prefix+\'_classification_report.json\')).read_text()),classification_report(y,p,labels=range(6),target_names=LABELS,output_dict=True,zero_division=0))\n cm=confusion_matrix(y,p,labels=range(6));conf=json.loads((folder/(prefix+\'_confusions.json\')).read_text());assert conf[\'labels\']==LABELS and np.array_equal(conf[\'raw\'],cm)\n norm=np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(1,keepdims=True)!=0)\n assert np.allclose(conf[\'row_normalized\'],norm,atol=1e-12,rtol=0)\n return {\'status\':\'passed_independent_csv_verification\',\'split\':split,\'metrics\':metrics,\'prediction_sha256\':sha(folder/(prefix+\'_predictions.csv\')),\'exact_order_ids_labels\':True,\'probability_ce_confusion_checks\':True}\n')
p=base/"registered_source"/'scripts/train_usable_multimodal_v2.py';p.parent.mkdir(parents=True,exist_ok=True);p.write_text('"""Approved usable-cohort six-way multimodal train/validate executable; NEVER runs test.\n\nLaunch requires exact approved faculty usable paired data and coverage. --stage part1 hard-stops\nat epoch2. Part2 resumes the same four-epoch schedule, with final test implemented\nseparately against a frozen validation-selected checkpoint. No local real training.\n"""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport signal\nimport sys\nimport time\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\',\'false\')\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\',\':4096:8\')\nimport numpy as np\nimport pandas as pd\nimport torchvision\nimport transformers\nimport PIL\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader,Dataset,Subset\nfrom torchvision import transforms as T\nfrom torchvision.transforms import functional as F\nfrom torchvision.models import resnet50,ResNet50_Weights\nfrom transformers import AutoTokenizer,AutoConfig,BertModel,get_linear_schedule_with_warmup\nfrom sklearn.metrics import accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix,f1_score\nfrom usable_cohort_v2 import load_input, COHORT, PROTOCOL, COUNTS, REGISTRATION_SHA\nfrom multimodal_recovery_v1 import SCHEMA,atomic_json,sha,stable_seed,order,crop,capture_rng,restore_rng,map_text_encoder,save_generation,load_generation\n\nLABELS=[\'True\',\'Satire/Parody\',\'Misleading Content\',\'Imposter Content\',\'False Connection\',\'Manipulated Content\']\nSTOP=False\n\ndef utc():return datetime.now(timezone.utc).isoformat()\n\ndef request_stop(*_):\n global STOP\n STOP=True\n\nclass MaximumFusion(nn.Module):\n def __init__(self,text,image):\n  super().__init__();self.text=text;self.image=image\n  self.text_projection=nn.Sequential(nn.Linear(text.config.hidden_size,512),nn.LayerNorm(512))\n  self.image_projection=nn.Sequential(nn.Linear(2048,512),nn.LayerNorm(512))\n  self.classifier=nn.Sequential(nn.Linear(512,256),nn.GELU(),nn.Dropout(.2),nn.Linear(256,6))\n def forward(self,input_ids,attention_mask,images):\n  t=self.text(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state[:,0]\n  v=self.image(images)\n  return self.classifier(torch.maximum(self.text_projection(t),self.image_projection(v)))\n\nclass Pairs(Dataset):\n def __init__(self,frame,root,tokenizer,cfg,training,packed_index=None,packed_index_sha256=None):\n  self.frame=frame.reset_index(drop=True);self.root=Path(root).resolve();self.cfg=cfg;self.training=training;self.epoch=1\n  self.store=None\n  if packed_index is not None:\n   from packed_image_io_v3 import ArchiveStore\n   if not packed_index_sha256:raise ValueError(\'Pinned packed index checksum required\')\n   self.store=ArchiveStore(self.root,packed_index,packed_index_sha256)\n  self.tokens={\'input_ids\':[],\'attention_mask\':[]}\n  for start in range(0,len(frame),8192):\n   chunk=tokenizer(frame.clean_title.iloc[start:start+8192].tolist(),truncation=True,max_length=cfg[\'max_length\'],padding=False)\n   for k in self.tokens:self.tokens[k].extend(chunk[k])\n def __len__(self):return len(self.frame)\n def __getitem__(self,index):\n  row=self.frame.iloc[index]\n  try:\n   if self.store is None:\n    p=(self.root/row.image_path).resolve()\n    if not p.is_relative_to(self.root):raise ValueError(\'Image path escapes root: \'+row.id)\n    source=p\n   else:source=io.BytesIO(self.store.read(row))\n   with Image.open(source) as im:\n    im=im.convert(\'RGB\')\n    if self.training:\n     i,j,h,w=crop(im,self.cfg[\'seed\'],self.epoch,row.id)\n     im=F.resized_crop(im,i,j,h,w,[224,224],T.InterpolationMode.BILINEAR,antialias=True)\n    else:im=F.center_crop(F.resize(im,232,T.InterpolationMode.BILINEAR,antialias=True),[224,224])\n    tensor=F.normalize(F.to_tensor(im),[.485,.456,.406],[.229,.224,.225])\n  except Exception as e:raise RuntimeError(\'Image failure for \'+row.id) from e\n  return {**{k:v[index] for k,v in self.tokens.items()},\'images\':tensor,\'labels\':int(row[\'6_way_label\']),\'id\':row.id}\n\nclass Collate:\n def __init__(self,tokenizer):self.tokenizer=tokenizer\n def __call__(self,rows):\n  tokens=self.tokenizer.pad([{k:r[k] for k in [\'input_ids\',\'attention_mask\']} for r in rows],padding=True,pad_to_multiple_of=8,return_tensors=\'pt\')\n  return {**tokens,\'images\':torch.stack([r[\'images\'] for r in rows]),\'labels\':torch.tensor([r[\'labels\'] for r in rows]),\'ids\':[r[\'id\'] for r in rows]}\n\ndef verified_frames(root,coverage_path,packed_index=None):\n root=Path(root);coverage=json.loads(Path(coverage_path).read_text())\n if (coverage.get(\'image_gate_passed\') is not True or coverage.get(\'cohort_id\')!=COHORT\n     or coverage.get(\'cohort_registration_sha256\')!=REGISTRATION_SHA\n     or coverage.get(\'required\')!=682170 or coverage.get(\'usable\')!=682170\n     or coverage.get(\'split_counts\')!=COUNTS):raise ValueError(\'Approved full usable-image runtime gate required\')\n expected,reg=load_input(root)\n manifest=root/\'data/official_paired_v2/paired_manifest.csv\'\n if sha(manifest)!=coverage[\'manifest_sha256\']:raise ValueError(\'Manifest hash mismatch\')\n if packed_index is not None:\n  from packed_image_io_v3 import ArchiveStore\n  if (coverage.get(\'input_layout\')!=\'kaggle-packed-bin\' or\n      coverage.get(\'runtime_root\')!=str(root.resolve()) or\n      coverage.get(\'runtime_image_bytes_verified\') is not True):\n   raise ValueError(\'Approved full packed-image runtime gate required\')\n  ArchiveStore(root,packed_index,coverage[\'packed_index_sha256\'])\n frame=pd.read_csv(manifest,dtype=str,keep_default_na=False)\n if frame.id.duplicated().any() or set(frame.id)!=set(expected):raise ValueError(\'Wrong/duplicate paired cohort IDs\')\n canonical=pd.DataFrame(expected.values());canonical[\'6_way_label\']=canonical[\'6_way_label\'].astype(str)\n frames={}\n for split,count in COUNTS.items():\n  official=canonical[canonical.split.eq(split)];paired=frame[frame.split.eq(split)]\n  if len(paired)!=count or set(paired.id)!=set(official.id):raise ValueError(\'Wrong approved split membership/count\')\n  paired=paired.set_index(\'id\').loc[official.id].reset_index()\n  for key in [\'clean_title\',\'6_way_label\']:\n   if not np.array_equal(paired[key].to_numpy(),official[key].to_numpy()):raise ValueError(\'Official \'+key+\' differs\')\n  if set(paired[\'6_way_label\'])!=set(map(str,range(6))) or paired.clean_title.str.strip().eq(\'\').any():raise ValueError(\'Invalid text/labels\')\n  if packed_index is None:\n   for row in paired[[\'id\',\'image_path\',\'image_sha256\']].itertuples(index=False):\n    q=(root/row.image_path).resolve()\n    if not q.is_relative_to(root.resolve()) or not q.is_file():raise ValueError(\'Missing/escaping image \'+row.id)\n    if sha(q)!=row.image_sha256:raise ValueError(\'Runtime image hash mismatch \'+row.id)\n  if split!=\'test\':frames[split]=paired\n return frames,{\'manifest_sha256\':sha(manifest),\'coverage_sha256\':sha(coverage_path),\n  \'tsv_sha256\':{x[\'filename\']:x[\'sha256\'] for x in reg[\'original_files\']},\n  \'cohort_id\':COHORT,\'cohort_registration_sha256\':REGISTRATION_SHA,\n  \'excluded_ids_sha256\':reg[\'excluded_ids_sha256\'],\'split_counts\':COUNTS,\n  \'extraction_report_sha256\':reg[\'extraction_report_sha256\'],\n  \'storage_mode\':\'packed\' if packed_index is not None else \'unpacked\',\n  \'packed_index_sha256\':coverage.get(\'packed_index_sha256\') if packed_index is not None else None}\n\ndef measure(y,pred,ce):\n return {\'accuracy\':float(accuracy_score(y,pred)),\'micro_f1\':float(f1_score(y,pred,average=\'micro\')),\n  \'macro_f1\':float(f1_score(y,pred,average=\'macro\',labels=range(6),zero_division=0)),\n  \'weighted_f1\':float(f1_score(y,pred,average=\'weighted\',labels=range(6),zero_division=0)),\n  \'balanced_accuracy\':float(balanced_accuracy_score(y,pred)),\'loss\':float(np.mean(ce)),\'samples\':len(y)}\n\n@torch.inference_mode()\ndef evaluate(model,loader,device):\n model.eval();ids=[];ys=[];probabilities=[];losses=[]\n for batch in loader:\n  ids.extend(batch.pop(\'ids\'));labels=batch.pop(\'labels\').to(device)\n  with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n   logits=model(**{k:v.to(device,non_blocking=True) for k,v in batch.items()})\n  # Export full float32 CE rather than infer loss from rounded confidence.\n  loss=nn.functional.cross_entropy(logits.float(),labels,reduction=\'none\')\n  ys.extend(labels.cpu().tolist());losses.extend(loss.cpu().tolist());probabilities.extend(logits.float().softmax(-1).cpu().tolist())\n if len(ids)!=len(set(ids)) or not losses:raise ValueError(\'Duplicate/empty evaluation\')\n probs=np.array(probabilities);pred=probs.argmax(1)\n return measure(ys,pred,losses),pd.DataFrame({\'id\':ids,\'true_label\':ys,\'predicted_label\':pred,\'cross_entropy\':losses,**{\'p\'+str(i):probs[:,i] for i in range(6)}})\n\ndef export_eval(folder,epoch,values,frame):\n prefix=\'validation_epoch\'+str(epoch);path=folder/(prefix+\'_predictions.csv\');frame.to_csv(path,index=False,float_format=\'%.17g\')\n y=frame.true_label.to_numpy();pred=frame.predicted_label.to_numpy();cm=confusion_matrix(y,pred,labels=range(6))\n atomic_json(folder/(prefix+\'_metrics.json\'),values)\n atomic_json(folder/(prefix+\'_classification_report.json\'),classification_report(y,pred,labels=range(6),target_names=LABELS,zero_division=0,output_dict=True))\n atomic_json(folder/(prefix+\'_confusions.json\'),{\'labels\':LABELS,\'raw\':cm.tolist(),\'row_normalized\':np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(1,keepdims=True)!=0).tolist()})\n return {\'predictions_file\':path.name,\'predictions_sha256\':sha(path),\'metrics\':values,\'epoch\':epoch}\n\ndef better(values,best):\n return best is None or (values[\'macro_f1\'],-values[\'loss\'])>(best[\'metrics\'][\'macro_f1\'],-best[\'metrics\'][\'loss\'])\n\nclass Engine:\n """The actual loop is injectable for synthetic interrupted-resume regression."""\n def __init__(self,model,optimizer,scheduler,scaler,cfg,output,device,contract):\n  self.model=model;self.optimizer=optimizer;self.scheduler=scheduler;self.scaler=scaler;self.cfg=cfg;self.out=Path(output);self.out.mkdir(parents=True,exist_ok=True);self.device=device;self.contract=contract\n  self.state={\'epoch\':1,\'cursor\':0,\'phase\':\'train\',\'updates\':0,\'skips\':0,\'loss_sum\':0.,\'seen\':0,\'history\':[],\'best\':None,\'patience_reference\':-1.,\'stale\':0,\'early_stopped\':False}\n  self.last_save=time.monotonic()\n def save(self):\n  if any(p.grad is not None for p in self.model.parameters()):raise ValueError(\'Gradients not cleared\')\n  payload={\'schema\':SCHEMA,\'gradients_cleared\':True,\'cursor\':self.state[\'cursor\'],\'state\':self.state,\'model\':self.model.state_dict(),\'optimizer\':self.optimizer.state_dict(),\'scheduler\':self.scheduler.state_dict(),\'scaler\':self.scaler.state_dict(),\'rng\':capture_rng(),\'contract\':self.contract}\n  receipt=save_generation(self.out/\'recovery\',payload);self.last_save=time.monotonic()\n  atomic_json(self.out/\'runtime_recovery.json\',{\'utc\':utc(),\'state\':self.state,\'generation\':receipt,\'durability\':\'session-local only until independently exported and verified\'})\n  return receipt\n def resume(self):\n  p,receipt=load_generation(self.out/\'recovery\')\n  if p[\'contract\']!=self.contract:raise ValueError(\'Immutable config/source/input contract changed\')\n  for obj,key in [(self.model,\'model\'),(self.optimizer,\'optimizer\'),(self.scheduler,\'scheduler\'),(self.scaler,\'scaler\')]:obj.load_state_dict(p[key])\n  self.state=p[\'state\'];restore_rng(p[\'rng\']);self.optimizer.zero_grad(set_to_none=True)\n  if self.state[\'best\']:\n   path=self.out/self.state[\'best\'][\'weights_file\']\n   if sha(path)!=self.state[\'best\'][\'weights_sha256\']:raise ValueError(\'Selected weights missing/corrupt\')\n  return receipt\n def train_epoch(self,loader,total_rows,on_boundary=None):\n  self.model.train();self.optimizer.zero_grad(set_to_none=True);s=self.state;cfg=self.cfg\n  if s[\'cursor\']!=s[\'seen\']:raise ValueError(\'Consumed cursor mismatch\')\n  original_cursor=s[\'cursor\'];micro=cfg[\'micro_batch\'];acc=cfg[\'accumulation\'];boundaries=0\n  for batch in loader:\n   ids=batch.pop(\'ids\');labels=batch.pop(\'labels\').to(self.device);n=len(labels)\n   group_start=(s[\'cursor\']//(micro*acc))*(micro*acc);group_n=min(micro*acc,total_rows-group_start)\n   with torch.autocast(self.device.type,dtype=torch.float16,enabled=self.device.type==\'cuda\'):\n    logits=self.model(**{k:v.to(self.device,non_blocking=True) for k,v in batch.items()});loss=nn.functional.cross_entropy(logits,labels)\n   if not torch.isfinite(loss):raise RuntimeError(\'Nonfinite loss; no silent sample exclusion\')\n   self.scaler.scale(loss*n/group_n).backward();s[\'loss_sum\']+=float(loss.detach())*n;s[\'seen\']+=n;s[\'cursor\']+=n\n   boundary=s[\'cursor\']%(micro*acc)==0 or s[\'cursor\']==total_rows\n   if boundary:\n    self.scaler.unscale_(self.optimizer);nn.utils.clip_grad_norm_(self.model.parameters(),cfg[\'gradient_clip\'])\n    scale=self.scaler.get_scale();self.scaler.step(self.optimizer);self.scaler.update()\n    if self.scaler.get_scale()>=scale:self.scheduler.step();s[\'updates\']+=1\n    else:s[\'skips\']+=1\n    self.optimizer.zero_grad(set_to_none=True);boundaries+=1\n    if s[\'cursor\']==total_rows:s[\'phase\']=\'validation\'\n    if on_boundary and on_boundary(self,ids,boundaries):self.save();return False\n    stop=STOP or (self.out/\'STOP_REQUESTED\').exists()\n    due=(s[\'updates\']>0 and s[\'updates\']%cfg[\'checkpoint_updates\']==0) or time.monotonic()-self.last_save>=cfg[\'checkpoint_seconds\']\n    if due or stop or s[\'phase\']==\'validation\':self.save()\n    if stop:return False\n   if boundary and (boundaries%50==0 or s[\'cursor\']==total_rows):\n    atomic_json(self.out/\'status.json\',{\'utc\':utc(),\'state\':\'training\',\'epoch\':s[\'epoch\'],\'samples_seen\':s[\'cursor\'],\'batches_seen\':math.ceil(s[\'cursor\']/micro),\'optimizer_updates\':s[\'updates\'],\'amp_skips\':s[\'skips\'],\'loss\':s[\'loss_sum\']/s[\'seen\']})\n  if s[\'cursor\']!=total_rows:raise ValueError(\'Loader ended before full epoch\')\n  return True\n def finish_validation(self,values,proof):\n  s=self.state;epoch=s[\'epoch\']\n  if s[\'phase\']!=\'validation\':raise ValueError(\'Validation before training completion\')\n  if better(values,s[\'best\']):\n   name=\'selected_epoch%d_%s.pt\'%(epoch,uuid.uuid4().hex);path=self.out/name\n   with path.open(\'wb\') as f:torch.save(self.model.state_dict(),f);f.flush();os.fsync(f.fileno())\n   s[\'best\']={**proof,\'weights_file\':name,\'weights_sha256\':sha(path)}\n  s[\'history\'].append({\'epoch\':epoch,\'train_loss\':s[\'loss_sum\']/s[\'seen\'],\'validation\':values,\'updates\':s[\'updates\'],\'skips\':s[\'skips\']})\n  if values[\'macro_f1\']>=s[\'patience_reference\']+self.cfg[\'early_stop_delta\']:s[\'patience_reference\']=values[\'macro_f1\'];s[\'stale\']=0\n  else:s[\'stale\']+=1\n  s[\'early_stopped\']=epoch>=self.cfg[\'early_stop_min_epoch\'] and s[\'stale\']>=self.cfg[\'early_stop_patience\']\n  s.update(epoch=epoch+1,cursor=0,phase=\'train\',loss_sum=0.,seen=0);self.save();atomic_json(self.out/\'history.json\',s[\'history\'])\n\ndef build_loader(dataset,tokenizer,cfg,training,cursor=0):\n if training:\n  permutation=order(cfg[\'seed\'],dataset.epoch,len(dataset));data=Subset(dataset,permutation[cursor:].tolist())\n else:data=dataset\n generator=torch.Generator().manual_seed(stable_seed(cfg[\'seed\'],dataset.epoch,\'workers-\'+str(cursor)))\n return DataLoader(data,batch_size=cfg[\'micro_batch\'] if training else cfg[\'eval_batch\'],shuffle=False,num_workers=cfg[\'workers\'],generator=generator,collate_fn=Collate(tokenizer),pin_memory=True,persistent_workers=False)\n\ndef run(args):\n cfg=json.loads(Path(args.config).read_text())\n if cfg.get(\'protocol_id\')!=PROTOCOL or cfg.get(\'cohort_registration_sha256\')!=REGISTRATION_SHA:raise ValueError(\'Unregistered usable-cohort configuration\')\n if cfg[\'planned_epochs\']!=4 or cfg[\'seed\']!=42:raise ValueError(\'Unexpected registered budget/seed\')\n stop=args.stop_after_epoch\n if args.stage==\'part1\' and stop not in [1,2]:raise ValueError(\'Part1 hard limit is epoch2; no test in this executable\')\n if args.stage==\'part2\' and (not args.resume or stop not in [3,4]):raise ValueError(\'Part2 must resume same run\')\n frames,inputs=verified_frames(args.root,args.coverage,args.packed_index)\n if not torch.cuda.is_available():raise RuntimeError(\'Cloud CUDA required; local scientific training refused\')\n out=Path(args.output);out.mkdir(parents=True,exist_ok=args.resume)\n lock=out/\'trainer.lock\'\n try:fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)\n except FileExistsError:raise RuntimeError(\'Attempt lock exists; verify original process/job before removing it\')\n os.write(fd,str(os.getpid()).encode());os.close(fd)\n try:\n  source_files=[Path(__file__),Path(__file__).with_name(\'multimodal_recovery_v1.py\'),Path(__file__).with_name(\'usable_cohort_v2.py\')]\n  if args.packed_index is not None:source_files.append(Path(__file__).with_name(\'packed_image_io_v3.py\'))\n  versions={\'python\':sys.version,\'torch\':torch.__version__,\'torchvision\':torchvision.__version__,\'transformers\':transformers.__version__,\'numpy\':np.__version__,\'pillow\':PIL.__version__}\n  contract={\'config\':cfg,\'config_sha256\':sha(args.config),\'inputs\':inputs,\'versions\':versions,\'sources\':{p.name:sha(p) for p in source_files}}\n  protocol=out/\'protocol.json\'\n  if args.resume and json.loads(protocol.read_text())!=contract:raise ValueError(\'Run protocol mismatch\')\n  atomic_json(protocol,contract)\n  random.seed(cfg[\'seed\']);np.random.seed(cfg[\'seed\']);torch.manual_seed(cfg[\'seed\']);torch.cuda.manual_seed_all(cfg[\'seed\']);torch.backends.cudnn.benchmark=False;torch.backends.cuda.matmul.allow_tf32=False;torch.use_deterministic_algorithms(True,warn_only=True)\n  tokenizer=AutoTokenizer.from_pretrained(cfg[\'bert_checkpoint\'],revision=cfg[\'bert_revision\']);bcfg=AutoConfig.from_pretrained(cfg[\'bert_checkpoint\'],revision=cfg[\'bert_revision\'])\n  text=BertModel(bcfg,add_pooling_layer=False)\n  if not args.resume:\n   if sha(args.warmstart)!=cfg[\'warmstart_sha256\']:raise ValueError(\'Warmstart hash mismatch\')\n   atomic_json(out/\'warmstart_mapping.json\',map_text_encoder(torch.load(args.warmstart,map_location=\'cpu\',weights_only=True),text))\n  image=resnet50(weights=None if args.resume else ResNet50_Weights.IMAGENET1K_V2);image.fc=nn.Identity();model=MaximumFusion(text,image).cuda()\n  groups={}\n  for name,p in model.named_parameters():\n   family=\'bert\' if name.startswith(\'text.\') else \'image\' if name.startswith(\'image.\') else \'head\';lr=cfg[family+\'_lr\'];decay=0. if p.ndim<=1 else cfg[\'image_decay\' if family==\'image\' else \'bert_head_decay\'];groups.setdefault((lr,decay),[]).append(p)\n  opt=torch.optim.AdamW([{\'params\':p,\'lr\':lr,\'weight_decay\':wd} for (lr,wd),p in groups.items()],betas=(.9,.999),eps=1e-8)\n  budget=math.ceil(math.ceil(len(frames[\'train\'])/cfg[\'micro_batch\'])/cfg[\'accumulation\'])*cfg[\'planned_epochs\'];scheduler=get_linear_schedule_with_warmup(opt,math.ceil(budget*cfg[\'warmup_fraction\']),budget);scaler=torch.amp.GradScaler(\'cuda\')\n  if budget!=cfg[\'planned_update_boundaries\'] or math.ceil(budget*cfg[\'warmup_fraction\'])!=cfg[\'warmup_boundaries\']:raise ValueError(\'Registered cohort schedule mismatch\')\n  data={k:Pairs(f,args.root,tokenizer,cfg,k==\'train\',args.packed_index,inputs[\'packed_index_sha256\']) for k,f in frames.items()}\n  engine=Engine(model,opt,scheduler,scaler,cfg,out,torch.device(\'cuda\'),contract)\n  if args.resume:atomic_json(out/\'last_resume_verification.json\',engine.resume())\n  else:tokenizer.save_pretrained(out/\'tokenizer\');engine.save()\n  if args.stage==\'part2\' and len(engine.state[\'history\'])<2:raise ValueError(\'Part2 requires completed Part1 train/validation history\')\n  if args.stage==\'part1\' and len(engine.state[\'history\'])>2:raise ValueError(\'This attempt has progressed past Part1; preserve its later state\')\n  atomic_json(out/\'environment.json\',{\'utc\':utc(),\'python\':sys.version,\'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\'used_devices\':1,\'cuda_visible_devices\':os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\'planned_update_boundaries\':budget})\n  for sig in [signal.SIGTERM,signal.SIGINT]:signal.signal(sig,request_stop)\n  while engine.state[\'epoch\']<=stop and not engine.state[\'early_stopped\']:\n   if STOP or (out/\'STOP_REQUESTED\').exists():engine.save();return\n   epoch=engine.state[\'epoch\'];data[\'train\'].epoch=epoch\n   if engine.state[\'phase\']==\'train\':\n    if not engine.train_epoch(build_loader(data[\'train\'],tokenizer,cfg,True,engine.state[\'cursor\']),len(data[\'train\'])):return\n   values,pred=evaluate(model,build_loader(data[\'validation\'],tokenizer,cfg,False),torch.device(\'cuda\'))\n   if pred.id.tolist()!=frames[\'validation\'].id.tolist():raise ValueError(\'Validation ID/order mismatch\')\n   proof=export_eval(out,epoch,values,pred);engine.finish_validation(values,proof)\n  atomic_json(out/(\'part1_runtime_complete.json\' if args.stage==\'part1\' else \'training_runtime_complete.json\'),{\'utc\':utc(),\'epochs_completed\':len(engine.state[\'history\']),\'best\':engine.state[\'best\'],\'no_test_performed\':True,\'durability\':\'Runtime complete only; not accepted until independent verification/export/Git receipt\'})\n finally:lock.unlink(missing_ok=True)\n\nif __name__==\'__main__\':\n ap=argparse.ArgumentParser();ap.add_argument(\'--root\',required=True,type=Path);ap.add_argument(\'--coverage\',required=True,type=Path);ap.add_argument(\'--packed-index\',type=Path);ap.add_argument(\'--config\',required=True,type=Path);ap.add_argument(\'--output\',required=True,type=Path);ap.add_argument(\'--warmstart\',required=True,type=Path);ap.add_argument(\'--stage\',choices=[\'part1\',\'part2\'],default=\'part1\');ap.add_argument(\'--stop-after-epoch\',type=int,default=2);ap.add_argument(\'--resume\',action=\'store_true\');run(ap.parse_args())\n')
p=base/"registered_source"/'scripts/multimodal_recovery_v1.py';p.parent.mkdir(parents=True,exist_ok=True);p.write_text('"""Recovery primitives for OFFICIAL-6W-MM-v1; not a full trainer/launch gate.\n\nNo network or GPU allocation. State serialization trusts only owned local files.\n"""\nimport hashlib\nimport json\nimport os\nimport random\nimport uuid\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torchvision import transforms as T\n\nSCHEMA=\'MM-RECOVERY-v1\'\n\ndef sha(path):\n h=hashlib.sha256()\n with open(path,\'rb\') as f:\n  for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\n\ndef stable_seed(seed,epoch,ident):\n return int.from_bytes(hashlib.sha256(json.dumps([SCHEMA,seed,epoch,str(ident)],separators=(\',\',\':\')).encode()).digest()[:8],\'big\')%(2**63-1)\n\ndef order(seed,epoch,count):\n return torch.randperm(count,generator=torch.Generator().manual_seed(stable_seed(seed,epoch,\'sampler\')))\n\ndef crop(image,seed,epoch,ident):\n # Local CPU RNG fork makes augmentation independent of worker/model RNG.\n with torch.random.fork_rng(devices=[]):\n  # torch.manual_seed also seeds CUDA; crop must never change model CUDA RNG.\n  torch.random.default_generator.manual_seed(stable_seed(seed,epoch,ident))\n  return T.RandomResizedCrop.get_params(image,(.8,1.),(.75,4/3))\n\ndef capture_rng():\n return {\'python\':random.getstate(),\'numpy\':np.random.get_state(),\'torch\':torch.get_rng_state(),\n         \'cuda\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\ndef restore_rng(state):\n random.setstate(state[\'python\']);np.random.set_state(state[\'numpy\']);torch.set_rng_state(state[\'torch\'].cpu())\n if state[\'cuda\']:torch.cuda.set_rng_state_all([x.cpu() for x in state[\'cuda\']])\n\ndef map_text_encoder(source,target):\n excluded={\'bert.pooler.dense.weight\',\'bert.pooler.dense.bias\',\'classifier.weight\',\'classifier.bias\'}\n if set(source)-excluded != {\'bert.\'+k for k in target.state_dict()}:raise ValueError(\'Unexpected warm-start keys\')\n mapped={k.removeprefix(\'bert.\'):v for k,v in source.items() if k not in excluded}\n for k,v in target.state_dict().items():\n  if v.shape!=mapped[k].shape or v.dtype!=mapped[k].dtype:raise ValueError(\'Warm-start shape/dtype \'+k)\n target.load_state_dict(mapped,strict=True)\n return {\'mapped_keys\':len(mapped),\'excluded_keys\':sorted(excluded),\'strict\':True}\n\ndef atomic_json(path,value):\n path=Path(path);tmp=path.with_name(path.name+\'.tmp-\'+uuid.uuid4().hex)\n with tmp.open(\'w\') as f:json.dump(value,f,indent=2,allow_nan=False);f.write(\'\\n\');f.flush();os.fsync(f.fileno())\n os.replace(tmp,path)\n\ndef save_generation(folder,payload):\n folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n if payload.get(\'schema\')!=SCHEMA or not payload.get(\'gradients_cleared\'):raise ValueError(\'Not a safe recovery boundary\')\n generation=uuid.uuid4().hex;path=folder/(generation+\'.pt\');tmp=path.with_suffix(\'.tmp\')\n with tmp.open(\'wb\') as f:torch.save(payload,f);f.flush();os.fsync(f.fileno())\n os.replace(tmp,path)\n # Reload before committing pointer. This alone is scratch persistence, not cloud durability.\n check=torch.load(path,map_location=\'cpu\',weights_only=False)\n if check[\'schema\']!=SCHEMA or check[\'cursor\']!=payload[\'cursor\']:raise ValueError(\'Reload failed\')\n item={\'file\':path.name,\'sha256\':sha(path),\'bytes\':path.stat().st_size,\'cursor\':payload[\'cursor\']}\n pointer=folder/\'generations.json\';old=json.loads(pointer.read_text()) if pointer.exists() else {\'generations\':[]}\n generations=[item]+old[\'generations\'][:1];atomic_json(pointer,{\'schema\':SCHEMA,\'generations\':generations})\n # Bound scratch space only after the new pointer is committed. Never sweep\n # unrelated files or uncommitted/corrupt artifacts retained for diagnosis.\n for retired in old[\'generations\'][1:]:\n  candidate=folder/retired[\'file\']\n  if candidate.parent.resolve()==folder.resolve() and len(candidate.stem)==32 and all(c in \'0123456789abcdef\' for c in candidate.stem):\n   candidate.unlink(missing_ok=True)\n return item\n\ndef load_generation(folder):\n folder=Path(folder);pointer=json.loads((folder/\'generations.json\').read_text());errors=[]\n for generation in pointer[\'generations\']:\n  try:\n   path=folder/generation[\'file\']\n   if path.parent.resolve()!=folder.resolve() or sha(path)!=generation[\'sha256\']:raise ValueError(\'Hash/path mismatch\')\n   payload=torch.load(path,map_location=\'cpu\',weights_only=False)\n   if payload[\'schema\']!=SCHEMA or payload[\'cursor\']!=generation[\'cursor\']:raise ValueError(\'Schema/cursor mismatch\')\n   return payload,{\'loaded\':generation,\'fallback_errors\':errors}\n  except (OSError,ValueError,RuntimeError,KeyError) as e:errors.append(str(e))\n raise ValueError(\'No usable recovery generation: \'+repr(errors))\n')
p=base/"registered_source"/'scripts/usable_cohort_v2.py';p.parent.mkdir(parents=True,exist_ok=True);p.write_text('"""Pinned user-approved available-image cohort; originals remain unchanged.\n\nThis verifies row membership only. It never certifies image bytes or runs models.\n"""\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter\nfrom pathlib import Path\n\nREGISTRATION = \'MM_PART1_USABLE_COHORT_REGISTRATION_20261024.json\'\nREGISTRATION_SHA = \'3fe0b622958a1eee6732789d4f7acf6b450c0e3bda87312e5cbb6500fb405ce5\'\nCOHORT = \'FACULTY-6W-USABLE-682170-v1\'\nPROTOCOL = \'OFFICIAL-6W-MM-USABLE-v2\'\nCOUNTS = {\'train\': 563601, \'validation\': 59299, \'test\': 59270}\nREQUIRED = 682170\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(4 * 1024**2), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef registration(path):\n    if sha(path) != REGISTRATION_SHA:\n        raise ValueError(\'Approved cohort registration hash mismatch\')\n    value = json.loads(Path(path).read_text())\n    if (value.get(\'approved\') is not True or value.get(\'cohort_id\') != COHORT\n            or value.get(\'protocol_id\') != PROTOCOL or value.get(\'usable_rows\') != REQUIRED\n            or value.get(\'split_counts\') != COUNTS):\n        raise ValueError(\'Wrong approved cohort\')\n    return value\n\n\ndef load_cohort(tsv_dir, registration_path, extraction_path, approval_path):\n    reg = registration(registration_path)\n    if sha(approval_path) != reg[\'approval_sha256\']:\n        raise ValueError(\'User approval receipt mismatch\')\n    if json.loads(Path(approval_path).read_text()).get(\'status\') != \'approved_by_user\':\n        raise ValueError(\'User approval missing\')\n    if sha(extraction_path) != reg[\'extraction_report_sha256\']:\n        raise ValueError(\'Original extraction report hash mismatch\')\n    report = json.loads(Path(extraction_path).read_text())\n    if (report.get(\'source_id\') != reg[\'archive_source_id\']\n            or report.get(\'compressed_archive_sha256\') != reg[\'archive_sha256\']\n            or report.get(\'compressed_archive_bytes\') != reg[\'archive_bytes\']\n            or report.get(\'source_crc_consumed\') is not True\n            or report.get(\'required\') != reg[\'original_rows\']\n            or report.get(\'usable\') != REQUIRED\n            or report.get(\'image_gate_passed\') is not False):\n        raise ValueError(\'Original all-row source evidence differs; never rewrite its failed gate\')\n    failures = report[\'decode_failures\']\n    excluded = {row[\'id\'] for row in failures}\n    fingerprint = hashlib.sha256((\'\\n\'.join(sorted(excluded)) + \'\\n\').encode()).hexdigest()\n    if (len(failures) != len(excluded) or len(excluded) != reg[\'excluded_rows\']\n            or set(report[\'missing_ids\']) != excluded or fingerprint != reg[\'excluded_ids_sha256\']):\n        raise ValueError(\'Exclusions differ from exact approved 491 IDs\')\n    all_ids, found, expected = set(), set(), {}\n    counts, failed_counts, failed_classes = Counter(), Counter(), Counter()\n    classes = {key: Counter() for key in COUNTS}\n    for item in reg[\'original_files\']:\n        path = Path(tsv_dir) / item[\'filename\']\n        if sha(path) != item[\'sha256\']:\n            raise ValueError(\'Original faculty TSV hash mismatch: \' + item[\'filename\'])\n        count = 0\n        with path.open(newline=\'\') as stream:\n            for row in csv.DictReader(stream, delimiter=\'\\t\'):\n                ident = row[\'id\']; label = int(row[\'6_way_label\']); count += 1\n                if ident in all_ids or not ident.isalnum() or label not in range(6) or not row[\'clean_title\'].strip():\n                    raise ValueError(\'Invalid/duplicate original row\')\n                all_ids.add(ident)\n                split = item[\'split\']\n                if ident in excluded:\n                    found.add(ident); failed_counts[split] += 1; failed_classes[str(label)] += 1\n                else:\n                    expected[ident] = {\'id\': ident, \'split\': split,\n                                       \'clean_title\': row[\'clean_title\'], \'6_way_label\': label}\n                    counts[split] += 1; classes[split][str(label)] += 1\n        if count != item[\'rows\']:\n            raise ValueError(\'Original row count mismatch\')\n    if (len(all_ids) != reg[\'original_rows\'] or found != excluded or len(expected) != REQUIRED\n            or dict(counts) != COUNTS or dict(failed_counts) != reg[\'excluded_split_counts\']\n            or dict(failed_classes) != reg[\'excluded_class_counts\']\n            or {key: dict(value) for key, value in classes.items()} != reg[\'class_counts\']):\n        raise ValueError(\'Approved cohort membership/count/label audit failed\')\n    return expected, reg\n\n\ndef load_input(root):\n    root = Path(root)\n    reg = registration(root / \'results\' / REGISTRATION)\n    return load_cohort(root / \'data/official_multimodal_v2\', root / \'results\' / REGISTRATION,\n                       root / \'results\' / reg[\'extraction_report_file\'],\n                       root / \'results\' / reg[\'approval_file\'])\n')
p=base/"registered_source"/'scripts/packed_image_io_v3.py';p.parent.mkdir(parents=True,exist_ok=True);p.write_text('"""Lossless TAR image storage adapter. Does not change pixels or research rows.\n\nBuild a bounded index while strictly hashing/decoding every member. Read only\nthe pinned byte range during training. No image extraction or model execution.\n"""\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport sqlite3\nimport tarfile\nimport shutil\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom collections import OrderedDict\nfrom PIL import Image\n\n_INDEX_CACHE = {}\n_ARCHIVE_CACHE = {}\n\n\ndef cache_raw_archives(root, proofs, cache, minimum_free=120 * 1024**3):\n    """Copy exact source archives into bounded cloud scratch; never re-encode.\n\n    A complete SHA proof, not a filename or prior receipt, authorizes reuse.\n    Originals remain read-only. Temporary writes are owned by this cache only.\n    """\n    root, cache = Path(root).resolve(), Path(cache).resolve()\n    total = sum(size for size, digest in proofs.values())\n    if not proofs or total > 100 * 1024**3:\n        raise ValueError(\'Raw archive cache exceeds registered storage bound\')\n    cache.mkdir(parents=True, exist_ok=True)\n    if shutil.disk_usage(cache).free < minimum_free:\n        raise ValueError(\'Insufficient cloud scratch for verified raw archive cache\')\n    started = time.monotonic()\n\n    def copy_one(item):\n        name, (size, digest) = item\n        source, target = root / name, cache / name\n        if source.parent != root or target.parent != cache or not name.endswith(\'.bin\'):\n            raise ValueError(\'Unsafe raw archive cache name\')\n        if source.stat().st_size != size:\n            raise ValueError(\'Source raw archive size changed\')\n        if target.exists():\n            if target.stat().st_size != size or sha(target) != digest:\n                raise ValueError(\'Preserve conflicting cloud raw archive cache\')\n            return {\'file\': name, \'bytes\': size, \'sha256\': digest, \'reused\': True}\n        temporary = target.with_suffix(\'.copy-\' + str(os.getpid()))\n        if temporary.exists():\n            raise ValueError(\'Preserve interrupted raw archive copy for inspection\')\n        h, count = hashlib.sha256(), 0\n        with source.open(\'rb\') as src, temporary.open(\'xb\') as dst:\n            while block := src.read(16 * 1024**2):\n                if time.monotonic() - started > 3600:\n                    raise TimeoutError(\'Raw archive staging exceeded one-hour bound\')\n                dst.write(block); h.update(block); count += len(block)\n            dst.flush(); os.fsync(dst.fileno())\n        if count != size or h.hexdigest() != digest:\n            raise ValueError(\'Raw archive cache copy SHA mismatch\')\n        os.replace(temporary, target)\n        print(json.dumps({\'state\': \'raw_archive_cached\', \'file\': name,\n                          \'bytes\': count, \'seconds\': time.monotonic()-started}), flush=True)\n        return {\'file\': name, \'bytes\': count, \'sha256\': digest, \'reused\': False}\n\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        copied = list(pool.map(copy_one, sorted(proofs.items())))\n    receipt = {\'state\': \'exact_raw_archive_cache_verified\', \'source\': str(root),\n               \'cache\': str(cache), \'bytes\': total, \'archives\': copied,\n               \'seconds\': time.monotonic()-started, \'pixels_changed\': False}\n    (cache / \'raw_archive_cache_receipt.json\').write_text(json.dumps(receipt, indent=2)+\'\\n\')\n    return cache\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef packed_name(archive):\n    p = PurePosixPath(archive)\n    if len(p.parts) != 1 or not archive.endswith(\'.tar\'):\n        raise ValueError(\'Expected a simple original TAR filename\')\n    return archive[:-4] + \'.bin\'\n\n\nclass HashReader:\n    def __init__(self, stream):\n        self.stream, self.digest, self.count = stream, hashlib.sha256(), 0\n\n    def read(self, size=-1):\n        block = self.stream.read(size)\n        self.digest.update(block)\n        self.count += len(block)\n        return block\n\n\ndef index_archive(root, archive, proof, rows, connection):\n    """One transaction per archive; caller commits a final index only after all."""\n    path = Path(root) / packed_name(archive)\n    if path.stat().st_size != proof[\'bytes\']:\n        raise ValueError(\'Packed archive size mismatch\')\n    expected = {r[\'id\']: r for r in rows}\n    if len(expected) != len(rows):\n        raise ValueError(\'Duplicate expected image IDs\')\n    seen = set()\n    with path.open(\'rb\') as source:\n        reader = HashReader(source)\n        with tarfile.open(fileobj=reader, mode=\'r|\') as container:\n            for member in container:\n                p = PurePosixPath(member.name)\n                if not member.isfile() or p.is_absolute() or \'..\' in p.parts:\n                    raise ValueError(\'Unexpected non-image or unsafe TAR member\')\n                ident = p.stem\n                row = expected.get(ident)\n                if row is None or ident in seen or row[\'source_shard\'] != archive:\n                    raise ValueError(\'Unexpected/duplicate image membership\')\n                if str(PurePosixPath(Path(archive).stem) / p) != row[\'image_path\']:\n                    raise ValueError(\'TAR member differs from original paired manifest\')\n                if member.size != int(row[\'bytes\']):\n                    raise ValueError(\'TAR image size differs from original paired manifest\')\n                raw = container.extractfile(member).read()\n                if len(raw) != member.size or hashlib.sha256(raw).hexdigest() != row[\'image_sha256\']:\n                    raise ValueError(\'Packed image SHA mismatch: \' + ident)\n                with Image.open(io.BytesIO(raw)) as image:\n                    image.load()\n                    image.convert(\'RGB\').load()\n                connection.execute(\'INSERT INTO images VALUES (?,?,?,?,?)\',\n                                   (ident, packed_name(archive), member.offset_data,\n                                    member.size, row[\'image_sha256\']))\n                seen.add(ident)\n        # Hash remaining TAR padding too, not merely the last image payload.\n        while reader.read(4 * 1024**2):\n            pass\n        if reader.count != proof[\'bytes\'] or reader.digest.hexdigest() != proof[\'sha256\']:\n            raise ValueError(\'Full original TAR SHA mismatch\')\n    if seen != set(expected):\n        raise ValueError(\'Incomplete archive image membership\')\n    return len(seen)\n\n\ndef create_index(root, shards, rows, output):\n    output = Path(output)\n    if output.exists():\n        raise ValueError(\'Preserve an existing index; explicit recovery required\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output.with_suffix(\'.building.sqlite\')\n    groups = {}\n    for row in rows:\n        groups.setdefault(row[\'source_shard\'], []).append(row)\n    if set(groups) != {s[\'file\'] for s in shards}:\n        raise ValueError(\'Archive set differs from manifest\')\n    count = 0\n    with sqlite3.connect(temporary) as db:\n        db.execute(\'PRAGMA synchronous=FULL\')\n        if not db.execute("SELECT name FROM sqlite_master WHERE type=\'table\' AND name=\'images\'").fetchone():\n            db.execute(\'CREATE TABLE images (id TEXT PRIMARY KEY, archive TEXT NOT NULL, offset INTEGER NOT NULL, size INTEGER NOT NULL, sha256 TEXT NOT NULL)\')\n            db.execute(\'CREATE TABLE archives (name TEXT PRIMARY KEY, bytes INTEGER NOT NULL, sha256 TEXT NOT NULL, images INTEGER NOT NULL)\')\n            db.commit()\n        if {x[0] for x in db.execute("SELECT name FROM sqlite_master WHERE type=\'table\'")} != {\'images\', \'archives\'}:\n            raise ValueError(\'Unrecognized incomplete index schema; preserve for inspection\')\n        if db.execute(\'PRAGMA integrity_check\').fetchone()[0] != \'ok\':\n            raise ValueError(\'Incomplete SQLite index failed integrity check\')\n        completed = {name: (size, digest, n) for name, size, digest, n in\n                     db.execute(\'SELECT name,bytes,sha256,images FROM archives\')}\n        if set(completed) - set(groups):\n            raise ValueError(\'Incomplete index contains an unexpected archive\')\n        for shard in shards:\n            name = shard[\'file\']\n            if name in completed:\n                size, digest, n = completed[name]\n                actual = list(db.execute(\'SELECT id,offset,size,sha256 FROM images WHERE archive=?\', (packed_name(name),)))\n                expected = {r[\'id\']: (int(r[\'bytes\']), r[\'image_sha256\']) for r in groups[name]}\n                if (size, digest, n) != (shard[\'bytes\'], shard[\'sha256\'], len(expected)) or len(actual) != n:\n                    raise ValueError(\'Incomplete index archive proof differs from pinned source\')\n                if any(ident not in expected or offset < 0 or (length, image_sha) != expected[ident]\n                       for ident, offset, length, image_sha in actual):\n                    raise ValueError(\'Incomplete index image rows differ from pinned manifest\')\n                path = Path(root) / packed_name(name)\n                if path.stat().st_size != size or sha(path) != digest:\n                    raise ValueError(\'Previously indexed archive bytes changed\')\n                count += n\n                print(json.dumps({\'state\': \'previous_packed_archive_reverified\', \'archive\': name,\n                                  \'verified_images\': count}), flush=True)\n                continue\n            try:\n                db.execute(\'BEGIN\')\n                n = index_archive(root, name, shard, groups[name], db)\n                db.execute(\'INSERT INTO archives VALUES (?,?,?,?)\', (name, shard[\'bytes\'], shard[\'sha256\'], n))\n                db.commit()\n            except BaseException:\n                db.rollback()\n                raise\n            count += n\n            print(json.dumps({\'state\': \'packed_archive_fully_verified\', \'archive\': shard[\'file\'],\n                              \'verified_images\': count}), flush=True)\n        assert db.execute(\'SELECT count(*) FROM images\').fetchone()[0] == count == len(rows)\n    with temporary.open(\'rb\') as stream:\n        os.fsync(stream.fileno())\n    os.replace(temporary, output)\n    return {\'images\': count, \'index_sha256\': sha(output), \'index\': str(output)}\n\n\nclass ArchiveStore:\n    def __init__(self, root, index, index_sha256):\n        self.root, self.source_index = Path(root).resolve(), Path(index).resolve()\n        if sha(self.source_index) != index_sha256:\n            raise ValueError(\'Archive index SHA mismatch\')\n        self.index = self.source_index\n        # Kaggle inputs use a remote filesystem. SQLite\'s tiny random page reads\n        # were the measured bottleneck; cache only this exact 108 MB index.\n        if self.source_index.is_relative_to(Path(\'/kaggle/input\')):\n            cache = Path(\'/kaggle/working/urop_verified_index_cache\')\n            cache.mkdir(parents=True, exist_ok=True)\n            local = cache / (index_sha256 + \'.sqlite\')\n            if not local.exists():\n                temporary = local.with_suffix(\'.tmp-\' + str(os.getpid()))\n                shutil.copyfile(self.source_index, temporary)\n                if sha(temporary) != index_sha256:\n                    raise ValueError(\'Copied index SHA mismatch\')\n                os.replace(temporary, local)\n            if sha(local) != index_sha256:\n                raise ValueError(\'Preserve conflicting cached index\')\n            self.index = local\n        key = (str(self.index), index_sha256)\n        if key not in _INDEX_CACHE:\n            with sqlite3.connect(\'file:\' + str(self.index) + \'?mode=ro&immutable=1\', uri=True) as db:\n                rows = {ident: (name, offset, size, digest) for ident, name, offset, size, digest in\n                        db.execute(\'SELECT id,archive,offset,size,sha256 FROM images\')}\n                proofs = {packed_name(name): (size, digest) for name, size, digest in db.execute(\'SELECT name,bytes,sha256 FROM archives\')}\n            _INDEX_CACHE[key] = rows, proofs\n        self.rows, proofs = _INDEX_CACHE[key]\n        self.sizes = {name: item[0] for name, item in proofs.items()}\n        if self.root.is_relative_to(Path(\'/kaggle/input\')):\n            cache_key = (str(self.root), index_sha256)\n            if cache_key not in _ARCHIVE_CACHE:\n                _ARCHIVE_CACHE[cache_key] = cache_raw_archives(\n                    self.root, proofs, \'/tmp/urop_verified_raw_archives/\' + index_sha256)\n            self.root = _ARCHIVE_CACHE[cache_key]\n        self.pid, self.db, self.handles = None, None, OrderedDict()\n\n    def __getstate__(self):\n        return {**self.__dict__, \'pid\': None, \'db\': None, \'handles\': OrderedDict()}\n\n    def read(self, row):\n        if self.pid != os.getpid():\n            for handle in self.handles.values():\n                handle.close()\n            self.handles, self.pid = OrderedDict(), os.getpid()\n        item = self.rows.get(row[\'id\'])\n        if item is None:\n            raise ValueError(\'Image missing from verified archive index\')\n        name, offset, size, digest = item\n        if name != packed_name(row[\'source_shard\']) or size != int(row[\'bytes\']) or digest != row[\'image_sha256\']:\n            raise ValueError(\'Archive index differs from pinned original manifest\')\n        if name not in self.sizes or offset < 0 or offset + size > self.sizes[name]:\n            raise ValueError(\'Invalid indexed archive byte range\')\n        if name not in self.handles:\n            path = (self.root / name).resolve()\n            if path.parent != self.root or path.stat().st_size != self.sizes[name]:\n                raise ValueError(\'Packed archive path/size changed\')\n            self.handles[name] = path.open(\'rb\', buffering=0)\n            if len(self.handles) > 64:\n                _, old = self.handles.popitem(last=False)\n                old.close()\n        stream = self.handles.pop(name)\n        self.handles[name] = stream\n        stream.seek(offset)\n        raw = stream.read(size)\n        if len(raw) != size or hashlib.sha256(raw).hexdigest() != digest:\n            raise ValueError(\'Indexed image bytes changed\')\n        return raw\n')
p=base/"registered_source"/'scripts/verify_usable_multimodal_v2.py';p.parent.mkdir(parents=True,exist_ok=True);p.write_text('"""Independently recompute full-validation exports and Part1 runtime recovery.\n\nProduces runtime acceptance only. Cloud/local durability and Git receipts are\nseparate checks; no training or test inference is performed here.\n"""\nimport argparse,csv,json,math\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,classification_report,confusion_matrix\nfrom usable_cohort_v2 import load_input, PROTOCOL, REGISTRATION_SHA\nfrom multimodal_recovery_v1 import atomic_json,sha,load_generation\n\ndef verify_export(folder,epoch,official):\n prefix=\'validation_epoch\'+str(epoch);p=Path(folder)/(prefix+\'_predictions.csv\');frame=pd.read_csv(p,dtype={\'id\':str})\n assert len(frame)==59299 and frame.id.is_unique\n assert frame.id.tolist()==official.id.tolist()\n assert np.array_equal(frame.true_label.to_numpy(),official[\'6_way_label\'].astype(int).to_numpy())\n probs=frame[[\'p\'+str(i) for i in range(6)]].to_numpy();ce=frame.cross_entropy.to_numpy();gold=frame.true_label.to_numpy();pred=frame.predicted_label.to_numpy()\n assert np.isfinite(probs).all() and np.isfinite(ce).all() and (ce>=0).all()\n assert (probs>=0).all() and (probs<=1).all() and np.allclose(probs.sum(1),1,atol=2e-6,rtol=0)\n assert np.array_equal(probs.argmax(1),pred)\n # CE is computed from original logits, so allow float32 softmax/CE rounding.\n gold_probability=probs[np.arange(len(gold)),gold];positive=gold_probability>0\n derived=-np.log(gold_probability[positive])\n assert np.allclose(derived,ce[positive],rtol=2e-5,atol=2e-5)\n # A float32 softmax can underflow to zero for extreme valid logits. The\n # exported CE remains finite; do not manufacture a loss by clipping to1e-308.\n assert (ce[~positive]>=-np.log(np.finfo(np.float32).tiny)).all()\n metrics={\'accuracy\':float(accuracy_score(gold,pred)),\'micro_f1\':float(f1_score(gold,pred,average=\'micro\')),\'macro_f1\':float(f1_score(gold,pred,average=\'macro\',labels=range(6),zero_division=0)),\'weighted_f1\':float(f1_score(gold,pred,average=\'weighted\',labels=range(6),zero_division=0)),\'balanced_accuracy\':float(balanced_accuracy_score(gold,pred)),\'loss\':float(ce.mean()),\'samples\':len(frame)}\n stored=json.loads((Path(folder)/(prefix+\'_metrics.json\')).read_text())\n for k,v in metrics.items():assert abs(stored[k]-v)<1e-10,k\n reports=json.loads((Path(folder)/(prefix+\'_classification_report.json\')).read_text())\n labels=json.loads((Path(folder)/(prefix+\'_confusions.json\')).read_text())[\'labels\']\n expected=classification_report(gold,pred,labels=range(6),target_names=labels,zero_division=0,output_dict=True)\n def compare(a,b):\n  if isinstance(a,dict):\n   assert set(a)==set(b)\n   for key in a:compare(a[key],b[key])\n  else:assert abs(a-b)<1e-10\n compare(reports,expected)\n conf=json.loads((Path(folder)/(prefix+\'_confusions.json\')).read_text());cm=confusion_matrix(gold,pred,labels=range(6));assert np.array_equal(conf[\'raw\'],cm)\n norm=cm/cm.sum(1,keepdims=True);assert np.allclose(conf[\'row_normalized\'],norm,atol=1e-12,rtol=0)\n return {\'epoch\':epoch,\'metrics\':metrics,\'predictions_sha256\':sha(p)}\n\ndef verify_part1(folder,root):\n folder=Path(folder);root=Path(root)\n protocol=json.loads((folder/\'protocol.json\').read_text());p=protocol[\'inputs\'][\'tsv_sha256\'][\'multimodal_validate.tsv\'];official_path=root/\'data/official_multimodal_v2/multimodal_validate.tsv\';assert sha(official_path)==p\n assert p==\'c70140ec376e083e3780b5b488c96c8933678ca5800139bfb78c92e95df6946e\'\n assert protocol[\'config\'][\'protocol_id\']==PROTOCOL and protocol[\'config\'][\'seed\']==42 and protocol[\'config\'][\'planned_epochs\']==4\n expected,reg=load_input(root)\n assert protocol[\'inputs\'][\'cohort_registration_sha256\']==REGISTRATION_SHA\n official=pd.DataFrame([r for r in expected.values() if r[\'split\']==\'validation\'])\n payload,recovery=load_generation(folder/\'recovery\');assert payload[\'contract\']==protocol\n state=payload[\'state\'];assert state[\'epoch\']==3 and state[\'cursor\']==0 and state[\'phase\']==\'train\';assert [h[\'epoch\'] for h in state[\'history\']]==[1,2]\n expected_boundaries=17613*2;assert state[\'updates\']+state[\'skips\']==expected_boundaries\n values=[verify_export(folder,e,official) for e in [1,2]]\n for stored,actual in zip(state[\'history\'],values):\n  for k,v in actual[\'metrics\'].items():assert abs(stored[\'validation\'][k]-v)<1e-10\n selected=max(values,key=lambda x:(x[\'metrics\'][\'macro_f1\'],-x[\'metrics\'][\'loss\'],-x[\'epoch\']))\n assert selected[\'epoch\']==state[\'best\'][\'epoch\']\n assert sha(folder/state[\'best\'][\'weights_file\'])==state[\'best\'][\'weights_sha256\']\n assert not list(folder.glob(\'test*_predictions.csv\')) and not list(folder.glob(\'test*_metrics.json\'))\n artifact_names=[\'protocol.json\',\'history.json\',\'recovery/generations.json\',state[\'best\'][\'weights_file\']]\n for generation in json.loads((folder/\'recovery/generations.json\').read_text())[\'generations\']:artifact_names.append(\'recovery/\'+generation[\'file\'])\n for epoch in [1,2]:artifact_names.extend(\'validation_epoch%d_%s\'%(epoch,suffix) for suffix in [\'predictions.csv\',\'metrics.json\',\'classification_report.json\',\'confusions.json\'])\n artifacts={n:{\'bytes\':(folder/n).stat().st_size,\'sha256\':sha(folder/n)} for n in artifact_names}\n result={\'status\':\'verified_runtime_only\',\'epochs_trained\':2,\'epochs_validated\':2,\'cohort_id\':reg[\'cohort_id\'],\'cohort_registration_sha256\':REGISTRATION_SHA,\'next_epoch\':3,\'no_test_performed\':True,\'validation\':values,\'selected_epoch\':selected[\'epoch\'],\'recovery\':recovery,\'artifacts\':artifacts,\'not_verified\':[\'Private cloud saved-version persistence\',\'Local retrieval of durable recovery\',\'Git backup and all context updates\',\'Actual GPU numerical replay regression\']}\n atomic_json(folder/\'part1_runtime_verification.json\',result);return result\n\nif __name__==\'__main__\':\n ap=argparse.ArgumentParser();ap.add_argument(\'--run\',type=Path,required=True);ap.add_argument(\'--root\',type=Path,required=True);a=ap.parse_args();print(json.dumps(verify_part1(a.run,a.root),indent=2))\n')
p=base/"registered_source"/'configs/multimodal_usable_6way_v2.json';p.parent.mkdir(parents=True,exist_ok=True);p.write_text('{\n  "protocol_id": "OFFICIAL-6W-MM-USABLE-v2",\n  "seed": 42,\n  "planned_epochs": 4,\n  "bert_checkpoint": "google-bert/bert-base-uncased",\n  "bert_revision": "86b5e0934494bd15c9632b12f734a8a67f723594",\n  "warmstart_sha256": "870bea50fa608dddd20421c21fd1786e244b1323e22734093214b87023666753",\n  "image_weights": "ResNet50_Weights.IMAGENET1K_V2",\n  "max_length": 128,\n  "micro_batch": 8,\n  "accumulation": 4,\n  "eval_batch": 32,\n  "workers": 2,\n  "bert_lr": 1e-05,\n  "image_lr": 0.0001,\n  "head_lr": 0.001,\n  "bert_head_decay": 0.01,\n  "image_decay": 0.0001,\n  "warmup_fraction": 0.1,\n  "gradient_clip": 1.0,\n  "checkpoint_updates": 1000,\n  "checkpoint_seconds": 600,\n  "early_stop_min_epoch": 3,\n  "early_stop_patience": 2,\n  "early_stop_delta": 0.001,\n  "determinism": "warn-only; stable seed/epoch/id CPU crop; explicit epoch permutation",\n  "selection": "validation macro-F1, lower loss, earlier epoch",\n  "test_policy": "No test evaluation in training executable",\n  "freeze_policy": "User-approvedusable-imagecohort;scientificrunfreezesonlyafterrealdata/cloudpreflightpasses",\n  "cohort_id": "FACULTY-6W-USABLE-682170-v1",\n  "cohort_registration_sha256": "3fe0b622958a1eee6732789d4f7acf6b450c0e3bda87312e5cbb6500fb405ce5",\n  "train_rows": 563601,\n  "validation_rows": 59299,\n  "test_rows_integrity_only": 59270,\n  "planned_update_boundaries": 70452,\n  "warmup_boundaries": 7046\n}\n')
driver=base/"part2_driver.py"
driver.write_text('"""Same scientific experiment: bounded epochs3-4 then frozen evaluation if time permits.\nNo token/OAuth, source change, new cohort, optimizer reset or test-driven selection.\nRun only in copied private Kaggle GPU notebook after current UI capacity/job check.\n"""\nfrom pathlib import Path\nimport datetime,hashlib,json,os,subprocess,sys,time,shutil,importlib.metadata,gc\nATTEMPT=\'OFFICIAL-6W-MM-USABLE-v2_seed42_20261027\'\nCOMMIT=\'342d287007b7cbe795031228bdb4c2c5b4ee70b6\'\nDEADLINE=datetime.datetime.fromisoformat(\'2026-10-27T19:35:00+00:00\')\nRETURN_DEADLINE=datetime.datetime.fromisoformat(\'2026-10-27T19:43:00+00:00\')\nBASE=Path(\'/kaggle/working/urop_mm_part2_20261027\')\nSOURCE_HASHES={\'scripts/train_usable_multimodal_v2.py\':\'7f7fb90deb08a5901d1ca8343b780f6d82dd42b3e278d51f333568435becee0e\',\'scripts/multimodal_recovery_v1.py\':\'2f5f67ce79289c69279040d21b483e28535895f5c99c63e0021da3eee6d3d1c8\',\'scripts/usable_cohort_v2.py\':\'22bff1eb0fa14d102ad07786f936c5c1ed1582ec7bd5cee2b8e8dc5eb3520092\',\'scripts/packed_image_io_v3.py\':\'a29e66d9c2962eb5bdfdce9abab02866fc77ad44aa5711ec8009a09dda7bb60d\',\'scripts/verify_usable_multimodal_v2.py\':\'bc314502833835a42bba8bab2d069074018bafc807106a3eb503386e2671139b\',\'configs/multimodal_usable_6way_v2.json\':\'ad8f86f86b1fef453ae3bccda0aeb13cc50b298dc0d5d502f87c9ac4b041b483\'}\ndef now():return datetime.datetime.now(datetime.timezone.utc)\ndef sha(p):\n h=hashlib.sha256()\n with Path(p).open(\'rb\') as f:\n  for b in iter(lambda:f.read(4*1024**2),b\'\'):h.update(b)\n return h.hexdigest()\ndef write(p,d):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_suffix(\'.tmp\');tmp.write_text(json.dumps(d,indent=2,allow_nan=False)+\'\\n\');os.replace(tmp,p)\ndef matching(name,digest):\n matches=[p for p in Path(\'/kaggle/input\').rglob(name) if p.is_file() and sha(p)==digest]\n assert matches,\'Missing pinned input \'+name\n # Multiple identical nested copies are harmless; exact bytes decide.\n return sorted(matches,key=lambda p:(len(p.parts),str(p)))[0]\ndef run(audit):\n start=time.monotonic();BASE.mkdir(parents=True,exist_ok=True)\n assert os.environ.get(\'CUDA_VISIBLE_DEVICES\')==\'0\'\n assert (now()-datetime.datetime.fromisoformat(audit[\'observed_utc\'])).total_seconds()<1200,\'Refresh actual UI capacity/active-jobs check\'\n assert audit[\'active_jobs\']==0 and audit[\'account\']==\'rocky62\' and audit[\'quota_used_hours\']+7<30 and audit[\'original_environment\']==\'2026-10-22\'\n assert (DEADLINE-now()).total_seconds()>12000,\'At least one full epoch plus saving margin must fit\'\n assert shutil.disk_usage(\'/kaggle/working\').free>12*1024**3\n assert not any(\'train_usable_multimodal_v2.py\' in p for p in subprocess.check_output([\'ps\',\'-eo\',\'args\'],text=True).splitlines()),\'Conflicting trainer\'\n src=BASE/\'registered_source\'\n import requests\n for relative,digest in SOURCE_HASHES.items():\n  p=src/relative;p.parent.mkdir(parents=True,exist_ok=True)\n  if not p.exists():\n   r=requests.get(\'https://raw.githubusercontent.com/panvenaidu/Fake-News-Detection/\'+COMMIT+\'/\'+relative,timeout=45);r.raise_for_status();assert hashlib.sha256(r.content).hexdigest()==digest;p.write_bytes(r.content)\n  assert sha(p)==digest\n # Only already-used package version may be restored, never silently upgrade contract.\n try:tv=importlib.metadata.version(\'transformers\')\n except importlib.metadata.PackageNotFoundError:tv=None\n if tv!=\'4.48.3\':subprocess.run([sys.executable,\'-m\',\'pip\',\'-q\',\'install\',\'transformers==4.48.3\'],check=True,timeout=600)\n prior=matching(\'cloud_job_runtime_result.json\',\'743f0ee306091cbbe699319d7c2c4a862f752ea54a76eec5e4a32374d0d6274f\').parent\n protocol=json.loads((prior/\'protocol.json\').read_text());assert sha(prior/\'protocol.json\')==\'bcb36efe3383c7ecfcc4c94b0c5943df570435bb6debe2dcb04d808a67502e43\'\n for name,digest in protocol[\'sources\'].items():assert sha(src/\'scripts\'/name)==digest\n assert {n:importlib.metadata.version(n) for n in [\'torch\',\'torchvision\',\'transformers\',\'numpy\',\'pillow\']}=={k:v for k,v in protocol[\'versions\'].items() if k!=\'python\'}\n assert sys.version==protocol[\'versions\'][\'python\'],\'Python contract mismatch\'\n coverage=matching(\'packed_runtime_coverage.json\',protocol[\'inputs\'][\'coverage_sha256\']);cov=json.loads(coverage.read_text());root=Path(cov[\'runtime_root\']);assert root.is_dir()\n index=matching(\'packed_image_index.sqlite\',protocol[\'inputs\'][\'packed_index_sha256\'])\n assert cov[\'dataset_version\']==1 and cov[\'usable\']==682170 and cov[\'image_gate_passed\'] and cov[\'split_counts\']=={\'train\':563601,\'validation\':59299,\'test\':59270}\n output=BASE/ATTEMPT;assert not output.exists(),\'Existing attempt requires explicit verified recovery, not overwrite\'\n assert not (prior/\'trainer.lock\').exists()\n assert sha(prior/\'recovery/2b57a4d8c3ca422f8c2e3773fdd845b7.pt\')==\'357c0707a80ef97f485cfe1824b537327aeb6810b89890055e6d18866e1f1819\'\n assert sha(prior/\'selected_epoch2_f9d1f5e980e74a1ba68c6432b9a6bba1.pt\')==\'fe3f78e27a77410884bddd2f0929e2fc4b9717424251adbbaa325441989dda40\'\n shutil.copytree(prior,output)\n sys.path.insert(0,str(src/\'scripts\'))\n import torch\n assert torch.cuda.is_available() and torch.cuda.device_count()==1 and torch.cuda.get_device_name(0)==\'Tesla T4\'\n from multimodal_recovery_v1 import load_generation\n payload,proof=load_generation(output/\'recovery\');assert payload[\'contract\']==protocol and payload[\'gradients_cleared\'];state=payload[\'state\']\n assert state[\'epoch\']==3 and state[\'cursor\']==0 and state[\'phase\']==\'train\' and len(state[\'history\'])==2 and state[\'updates\']==35211 and state[\'skips\']==15\n assert payload[\'scheduler\'][\'last_epoch\']==35211 and len(payload[\'rng\'][\'cuda\'])==1 and all(k in payload for k in [\'model\',\'optimizer\',\'scheduler\',\'scaler\',\'rng\']);assert payload[\'optimizer\'][\'state\'] and payload[\'scaler\']\n del payload;gc.collect()\n release={\'utc\':now().isoformat(),\'state\':\'verified_part2_resume_release\',\'audit\':audit,\'attempt\':ATTEMPT,\'prior_output\':str(prior),\'prior_recovery\':proof,\'coverage\':str(coverage),\'index\':str(index),\'input_root\':str(root),\'protocol_sha256\':sha(output/\'protocol.json\'),\'source_commit\':COMMIT,\'training_deadline_utc\':DEADLINE.isoformat(),\'final_test_only_after_training_complete\':True,\'gpu\':torch.cuda.get_device_name(0),\'cuda_visible_devices\':os.environ.get(\'CUDA_VISIBLE_DEVICES\')}\n write(output/\'part2_release.json\',release);print(json.dumps(release),flush=True)\n warmstarts=list(Path(\'/kaggle/input\').rglob(\'selected_epoch4.pt\'))\n # Resume does not consume --warmstart; the mandatory path is provenance only.\n warmstart=prior/\'selected_epoch2_f9d1f5e980e74a1ba68c6432b9a6bba1.pt\'\n command=[sys.executable,\'-u\',str(src/\'scripts/train_usable_multimodal_v2.py\'),\'--root\',str(root),\'--coverage\',str(coverage),\'--packed-index\',str(index),\'--config\',str(src/\'configs/multimodal_usable_6way_v2.json\'),\'--output\',str(output),\'--warmstart\',str(warmstart),\'--stage\',\'part2\',\'--stop-after-epoch\',\'4\',\'--resume\']\n requested=False;log=BASE/\'part2_training_console.log\'\n with log.open(\'x\') as stream:\n  proc=subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT,env=os.environ.copy())\n  while proc.poll() is None:\n   status=json.loads((output/\'status.json\').read_text()) if (output/\'status.json\').exists() else {\'state\':\'starting\'}\n   if now()>=DEADLINE and not requested:(output/\'STOP_REQUESTED\').touch();requested=True\n   heartbeat={\'utc\':now().isoformat(),\'trainer_alive\':True,\'pid\':proc.pid,\'safe_stop_requested\':requested,\'status\':status,\'remaining_task_compute_seconds\':(DEADLINE-now()).total_seconds()}\n   write(output/\'part2_heartbeat.json\',heartbeat);print(json.dumps(heartbeat),flush=True)\n   if now()>=RETURN_DEADLINE:\n    proc.terminate()\n    try:proc.wait(timeout=120)\n    except subprocess.TimeoutExpired:proc.kill();proc.wait()\n    break\n   time.sleep(30)\n assert proc.returncode==0,\'Trainer failed; preserve checkpoint and console before any retry\'\n payload,proof=load_generation(output/\'recovery\');state=payload[\'state\'];assert payload[\'contract\']==protocol and payload[\'scheduler\'][\'last_epoch\']==state[\'updates\']\n del payload;gc.collect()\n from usable_cohort_v2 import load_input\n import pandas as pd\n from verify_usable_multimodal_v2 import verify_export\n expected,_=load_input(root);official=pd.DataFrame([r for r in expected.values() if r[\'split\']==\'validation\'])\n values=[verify_export(output,h[\'epoch\'],official) for h in state[\'history\']]\n best=max(values,key=lambda v:(v[\'metrics\'][\'macro_f1\'],-v[\'metrics\'][\'loss\'],-v[\'epoch\']));assert best[\'epoch\']==state[\'best\'][\'epoch\'];assert sha(output/state[\'best\'][\'weights_file\'])==state[\'best\'][\'weights_sha256\']\n complete=state[\'epoch\']>4 or state[\'early_stopped\'];result={\'utc\':now().isoformat(),\'state\':\'registered_training_complete\' if complete else \'partial_part2_durable_runtime_checkpoint\',\'training_complete\':complete,\'epochs_validated\':len(values),\'next_epoch\':state[\'epoch\'],\'cursor\':state[\'cursor\'],\'phase\':state[\'phase\'],\'recovery\':proof,\'selected\':state[\'best\'],\'validation\':values,\'elapsed_seconds\':time.monotonic()-start,\'no_test_performed\':True,\'requires_private_saved_version_and_independent_local_readback\':True}\n write(output/\'part2_runtime_result.json\',result)\n if complete and (RETURN_DEADLINE-now()).total_seconds()>900:\n  evaluator=BASE/\'evaluate_frozen_multimodal_v1.py\';assert evaluator.is_file()\n  subprocess.run([sys.executable,\'-u\',str(evaluator),\'--run\',str(output),\'--root\',str(root),\'--index\',str(index)],env=dict(os.environ,PYTHONPATH=str(src/\'scripts\')+os.pathsep+str(BASE)),check=True,timeout=max(60,int((RETURN_DEADLINE-now()).total_seconds())-120))\n  result[\'no_test_performed\']=False;result[\'state\']=\'runtime_train_validation_test_verified\';write(output/\'part2_runtime_result.json\',result)\n print(json.dumps(result),flush=True)\n return result\n\nrun({\'observed_utc\': \'2026-10-27T14:13:50+00:00\', \'account\': \'rocky62\', \'active_jobs\': 0, \'active_jobs_source\': \'Kaggle Your Work: No Active Events, zero Active Events before draft creation\', \'quota_used_hours\': 6.433333333333334, \'quota_total_hours\': 30, \'quota_precision\': \'UI rounded minutes;06:26/30h\', \'original_environment\': \'2026-10-22\', \'accelerator\': \'GPU T4 x2, one used CUDA_VISIBLE_DEVICES=0\', \'input_version_id_observed\': 21277535, \'accepted_previous_kernel_version\': 362761315, \'task_started_utc\': \'2026-10-27T14:08:54+00:00\', \'task_hard_deadline_utc\': \'2026-10-27T20:08:54+00:00\'})\n')
env=dict(os.environ,CUDA_VISIBLE_DEVICES='0',CUBLAS_WORKSPACE_CONFIG=':4096:8')
subprocess.run([sys.executable,'-u',str(driver)],env=env,check=True,timeout=21000)
